# Salary Prediction by College Degree, Major, and Years of Experience

**Goal:** Train a scikit-learn model to predict individual salary given:
- `degree_level` — Associate, Bachelor, Master, Professional, or Doctoral
- `major` — field of study (from ~166 ACS categories)
- `years_since_grad` — proxy for work experience

**Data source:** [ACS PUMS 2023 (1-Year)](https://www.census.gov/programs-surveys/acs/microdata/access/2023.html)  
American Community Survey Public Use Microdata Sample — ~3.2M individual person records, free from the US Census Bureau.

---

**Sections:**
0. Setup & constants  
1. Data download & load  
2. Cleaning & filtering  
3. Feature engineering  
4. EDA  
5. Model training & evaluation  
6. Inference / prediction helper

## 0 — Setup & Constants

In [ ]:
import io
import os
import zipfile
import warnings
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OrdinalEncoder,
    OneHotEncoder,
    PolynomialFeatures,
    StandardScaler,
)
from sklearn.preprocessing import TargetEncoder

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:,.0f}".format)
sns.set_theme(style="whitegrid", palette="muted")

# ── ACS PUMS 2023 1-Year download URL ────────────────────────────────────────
DATA_URL = "https://www2.census.gov/programs-surveys/acs/data/pums/2023/1-Year/csv_pus.zip"
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

# Columns to load (keeps memory footprint small)
COLS = ["SCHL", "FOD1P", "AGEP", "WAGP", "ADJINC", "ESR", "PWGTP"]

# ── Degree level mapping (SCHL codes → readable labels) ──────────────────────
DEGREE_MAP = {
    20: "Associate",
    21: "Bachelor",
    22: "Master",
    23: "Professional",
    24: "Doctoral",
}

# Ordinal order for encoding
DEGREE_ORDER = ["Associate", "Bachelor", "Master", "Professional", "Doctoral"]

# Assumed graduation age by degree level (used to compute years_since_grad)
GRAD_AGE = {
    "Associate": 20,
    "Bachelor": 22,
    "Master": 24,
    "Professional": 27,
    "Doctoral": 29,
}

# ── FOD1P → major name mapping (ACS 2023 Data Dictionary excerpt) ────────────
# Full list: https://www2.census.gov/programs-surveys/acs/tech_docs/pums/data_dict/PUMS_Data_Dictionary_2023.pdf
FOD1P_LABELS = {
    1100: "General Agriculture",
    1101: "Agriculture Production and Management",
    1102: "Agricultural Economics",
    1103: "Animal Sciences",
    1104: "Food Science",
    1105: "Plant Science and Agronomy",
    1106: "Soil Science",
    1107: "Veterinary Medicine",
    1199: "Miscellaneous Agriculture",
    1301: "Environmental Science",
    1302: "Forestry",
    1303: "Natural Resources Management",
    1401: "Architecture",
    1501: "Area Ethnic and Civilization Studies",
    1901: "Communications",
    1902: "Journalism",
    1903: "Mass Media",
    1904: "Advertising and Public Relations",
    2001: "Criminal Justice and Police Science",
    2100: "Social Work",
    2101: "Family and Consumer Sciences",
    2201: "General Education",
    2202: "Educational Administration and Supervision",
    2203: "School Student Counseling",
    2204: "Elementary Education",
    2205: "Mathematics Teacher Education",
    2206: "Physical and Health Education Teaching",
    2207: "Early Childhood Education",
    2208: "Science and Computer Teacher Education",
    2209: "Secondary Teacher Education",
    2210: "Special Needs Education",
    2211: "Social Science or History Teacher Education",
    2212: "Teacher Education: Multiple Levels",
    2213: "Language and Drama Education",
    2214: "Art and Music Education",
    2299: "Miscellaneous Education",
    2301: "Library Science",
    2400: "General Engineering",
    2401: "Aerospace Engineering",
    2402: "Biological Engineering",
    2403: "Architectural Engineering",
    2404: "Chemical Engineering",
    2405: "Civil Engineering",
    2406: "Computer Engineering",
    2407: "Electrical Engineering",
    2408: "Engineering Mechanics Physics and Science",
    2409: "Environmental Engineering",
    2410: "Geological and Geophysical Engineering",
    2411: "Industrial and Manufacturing Engineering",
    2412: "Materials Engineering and Materials Science",
    2413: "Mechanical Engineering",
    2414: "Metallurgical Engineering",
    2415: "Mining and Mineral Engineering",
    2416: "Naval Architecture and Marine Engineering",
    2417: "Nuclear Engineering",
    2418: "Petroleum Engineering",
    2419: "Miscellaneous Engineering",
    2500: "General Engineering Technologies",
    2501: "Engineering and Industrial Management",
    2502: "Electrical Engineering Technology",
    2503: "Industrial Production Technologies",
    2504: "Mechanical Engineering Related Technologies",
    2505: "Miscellaneous Engineering Technologies",
    2601: "Linguistics and Comparative Language and Literature",
    2602: "French German Latin and Other Common Foreign Language Studies",
    2603: "Other Foreign Languages",
    2901: "Family and Consumer Sciences",
    3202: "Pre-Law and Legal Studies",
    3301: "English Language and Literature",
    3302: "Composition and Rhetoric",
    3401: "Liberal Arts",
    3402: "Humanities",
    3501: "Library Science",
    3600: "Biology",
    3601: "Biochemical Sciences",
    3602: "Botany",
    3603: "Molecular Biology",
    3604: "Ecology",
    3605: "Genetics",
    3606: "Microbiology",
    3607: "Pharmacology",
    3608: "Physiology",
    3609: "Zoology",
    3611: "Neuroscience",
    3699: "Miscellaneous Biology",
    3700: "Mathematics",
    3701: "Applied Mathematics",
    3702: "Statistics and Decision Science",
    3801: "Military Technologies",
    4000: "Multi/Interdisciplinary Studies",
    4001: "Intercultural and International Studies",
    4002: "Nutrition Sciences",
    4005: "Mathematics and Computer Science",
    4006: "Cognitive Science and Biopsychology",
    4007: "Interdisciplinary Social Sciences",
    4008: "Biological and Physical Sciences",
    4009: "Data Science and Data Analytics",
    4101: "Physical Fitness Parks Recreation and Leisure",
    4801: "Philosophy and Religious Studies",
    4901: "Theology and Religious Vocations",
    5000: "Physical Sciences",
    5001: "Astronomy and Astrophysics",
    5002: "Atmospheric Sciences and Meteorology",
    5003: "Chemistry",
    5004: "Geology and Earth Science",
    5005: "Geosciences",
    5006: "Oceanography",
    5007: "Physics",
    5008: "Materials Science",
    5098: "Military Technologies",
    5102: "Nuclear Industrial Radiology and Biological Technologies",
    5200: "General Business",
    5201: "Accounting",
    5202: "Actuarial Science",
    5203: "Business Management and Administration",
    5204: "Operations Logistics and E-Commerce",
    5205: "Business Economics",
    5206: "Marketing and Marketing Research",
    5207: "Finance",
    5208: "Human Resources and Personnel Management",
    5209: "International Business",
    5210: "Hospitality Management",
    5211: "Management Information Systems and Statistics",
    5299: "Miscellaneous Business and Medical Administration",
    5301: "Educational and Vocational Guidance Counseling",
    5401: "Economics",
    5402: "Anthropology and Archeology",
    5403: "Criminology",
    5404: "Geography",
    5405: "International Relations",
    5406: "Political Science and Government",
    5407: "Sociology",
    5408: "Urban Planning",
    5409: "Social Psychology",
    5411: "Human Services and Community Organization",
    5412: "History",
    5500: "General Social Sciences",
    5501: "Industrial and Organizational Psychology",
    5502: "Clinical Psychology",
    5503: "Counseling Psychology",
    5504: "Psychology",
    5505: "Educational Psychology",
    5506: "Human Development",
    5507: "Social Psychology",
    5599: "Miscellaneous Social Sciences",
    5601: "Construction Services",
    5701: "Electrical, Mechanical, and Precision Technologies",
    5901: "Transportation Sciences and Technologies",
    6000: "Fine Arts",
    6001: "Drama and Theater Arts",
    6002: "Music",
    6003: "Visual and Performing Arts",
    6004: "Commercial Art and Graphic Design",
    6005: "Film Video and Photographic Arts",
    6006: "Art History and Criticism",
    6007: "Studio Arts",
    6099: "Miscellaneous Fine Arts",
    6100: "General Medical and Health Services",
    6102: "Communication Disorders Sciences and Services",
    6103: "Health and Medical Administrative Services",
    6104: "Medical Assisting Services",
    6105: "Medical Technologies Technicians",
    6106: "Health and Medical Preparatory Programs",
    6107: "Nursing",
    6108: "Pharmacy Pharmaceutical Sciences and Administration",
    6109: "Treatment Therapy Professions",
    6110: "Community and Public Health",
    6199: "Miscellaneous Health Medical Professions",
    6200: "General Business",
    6201: "Computer Science",
    6202: "Information Science",
    6203: "Computer Administration Management and Security",
    6204: "Computer Networking and Telecommunications",
    6206: "Computer and Information Systems",
    6207: "Computer Programming and Data Processing",
    6209: "Computer Information Technology",
    6210: "Computer and Information Systems",
    6299: "Miscellaneous Computers and Mathematics",
}

# ── Major category groupings (FOD1P → broad bucket) ─────────────────────────
# Maps FOD1P code → one of ~15 broad fields
def _make_major_category_map():
    cat = {}
    for code in FOD1P_LABELS:
        if 2400 <= code <= 2599:
            cat[code] = "Engineering"
        elif code in (6201, 6202, 6203, 6204, 6206, 6207, 6209, 6210, 6299, 4009, 4005):
            cat[code] = "Computer Science & IT"
        elif 5200 <= code <= 5299:
            cat[code] = "Business"
        elif code == 5401:
            cat[code] = "Economics"
        elif 5400 <= code <= 5412:
            cat[code] = "Social Sciences"
        elif 5500 <= code <= 5599:
            cat[code] = "Psychology"
        elif 2200 <= code <= 2299 or code == 2301:
            cat[code] = "Education"
        elif 6100 <= code <= 6199:
            cat[code] = "Health & Medicine"
        elif 3600 <= code <= 3699:
            cat[code] = "Biology & Life Sciences"
        elif 5000 <= code <= 5099 or code in (3700, 3701, 3702):
            cat[code] = "Physical Sciences & Math"
        elif 1100 <= code <= 1199 or code in (1301, 1302, 1303):
            cat[code] = "Agriculture & Environment"
        elif 6000 <= code <= 6099:
            cat[code] = "Arts & Design"
        elif code in (3301, 3302, 2601, 2602, 2603, 4801, 4901):
            cat[code] = "Humanities & Languages"
        elif 1901 <= code <= 1904:
            cat[code] = "Communications & Media"
        else:
            cat[code] = "Other"
    return cat

MAJOR_CATEGORY_MAP = _make_major_category_map()

print("Setup complete.")
print(f"  Degree levels: {DEGREE_ORDER}")
print(f"  FOD1P entries mapped: {len(FOD1P_LABELS)}")
print(f"  Major categories: {sorted(set(MAJOR_CATEGORY_MAP.values()))}")

## 1 — Data Download & Load

The ACS PUMS 2023 1-Year file contains ~3.2M person records split across two CSV files inside `csv_pus.zip` (~400 MB compressed).

**Note on WAGP top-coding:** The Census Bureau replaces wages above a state-specific threshold with the mean of all top-coded values. This compresses the upper tail of the salary distribution but does not affect the vast majority of respondents.

In [ ]:
def download_acs_pums(url: str, dest_dir: str) -> list[str]:
    """
    Download and extract ACS PUMS zip file.
    Returns list of extracted CSV file paths.
    Skips download if files already exist.
    """
    zip_path = os.path.join(dest_dir, "csv_pus.zip")
    csv_a = os.path.join(dest_dir, "psam_pusa.csv")
    csv_b = os.path.join(dest_dir, "psam_pusb.csv")

    if os.path.exists(csv_a) and os.path.exists(csv_b):
        print("CSV files already exist — skipping download.")
        return [csv_a, csv_b]

    print(f"Downloading ACS PUMS 2023 from Census Bureau...")
    print(f"URL: {url}")
    print("(~400 MB — this may take several minutes on a slow connection)\n")

    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        downloaded = 0
        with open(zip_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):  # 1 MB chunks
                f.write(chunk)
                downloaded += len(chunk)
                if total:
                    pct = downloaded / total * 100
                    print(f"\r  {downloaded / 1e6:.0f} MB / {total / 1e6:.0f} MB ({pct:.1f}%)", end="")
    print("\nDownload complete. Extracting...")

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dest_dir)
    print("Extraction complete.")

    return [csv_a, csv_b]


def load_pums(csv_paths: list[str], cols: list[str]) -> pd.DataFrame:
    """Load and concatenate PUMS person CSV files, selecting only needed columns."""
    parts = []
    for path in csv_paths:
        print(f"  Loading {os.path.basename(path)}...", end=" ")
        df = pd.read_csv(path, usecols=cols, low_memory=False)
        print(f"{len(df):,} rows")
        parts.append(df)
    combined = pd.concat(parts, ignore_index=True)
    print(f"\nCombined: {len(combined):,} rows, {combined.shape[1]} columns")
    return combined


# Download (skips if already present)
csv_files = download_acs_pums(DATA_URL, DATA_DIR)

# Load only the columns we need
raw = load_pums(csv_files, COLS)
raw.head()

## 2 — Cleaning & Filtering

In [ ]:
def clean_pums(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply filters and transformations to ACS PUMS person data.

    Filters:
      - Degree holders (SCHL 20–24)
      - Employed civilians (ESR 1 or 2)
      - Positive wages (WAGP > 0)
      - Working age (22–65)

    Transformations:
      - Inflation-adjust wages: adj_wage = WAGP * ADJINC / 1_000_000
      - Map SCHL → degree_level string
      - Map FOD1P → major_name and major_category
      - Fill missing FOD1P (associate holders + unreported) with "Unknown"
    """
    d = df.copy()

    # Coerce numeric columns
    for col in ["SCHL", "FOD1P", "AGEP", "WAGP", "ADJINC", "ESR", "PWGTP"]:
        d[col] = pd.to_numeric(d[col], errors="coerce")

    n0 = len(d)

    # Filter: degree holders only
    d = d[d["SCHL"].isin(DEGREE_MAP.keys())]
    print(f"After degree filter:      {len(d):>9,}  (dropped {n0 - len(d):,})")

    # Filter: employed civilians
    d = d[d["ESR"].isin([1, 2])]
    print(f"After employment filter:  {len(d):>9,}")

    # Filter: positive wages
    d = d[d["WAGP"] > 0]
    print(f"After wage > 0 filter:    {len(d):>9,}")

    # Filter: working age 22–65
    d = d[(d["AGEP"] >= 22) & (d["AGEP"] <= 65)]
    print(f"After age 22–65 filter:   {len(d):>9,}")

    # Inflation-adjust wages (ADJINC is stored as int × 1,000,000)
    d["adj_wage"] = d["WAGP"] * d["ADJINC"] / 1_000_000

    # Map degree level
    d["degree_level"] = d["SCHL"].map(DEGREE_MAP)

    # Map FOD1P → major name (NaN for associate or unreported)
    d["FOD1P_int"] = d["FOD1P"].astype("Int64")  # nullable int for mapping
    d["major_name"] = d["FOD1P_int"].map(FOD1P_LABELS).fillna("Unknown")
    d["major_category"] = d["FOD1P_int"].map(MAJOR_CATEGORY_MAP).fillna("Other")

    # Fill missing FOD1P with sentinel 0 for TargetEncoder (needs non-null)
    d["FOD1P_enc"] = d["FOD1P"].fillna(0).astype(int)

    return d.reset_index(drop=True)


df = clean_pums(raw)

print(f"\nFinal dataset: {len(df):,} rows")
print(f"\nDegree distribution:")
print(df["degree_level"].value_counts())
print(f"\nSample rows:")
df[["degree_level", "major_name", "major_category", "AGEP", "adj_wage", "PWGTP"]].head(10)

## 3 — Feature Engineering

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add model-ready features:
      - years_since_grad: AGEP minus expected graduation age, clipped [0, 45]
      - log_wage: log1p-transformed inflation-adjusted wage (regression target)

    Sanity checks are included at the end.
    """
    d = df.copy()

    # years_since_grad: age minus expected graduation age for degree type
    d["expected_grad_age"] = d["degree_level"].map(GRAD_AGE)
    d["years_since_grad"] = (d["AGEP"] - d["expected_grad_age"]).clip(lower=0, upper=45)

    # Log-transform target (improves linearity, handles right skew)
    d["log_wage"] = np.log1p(d["adj_wage"])

    # Sanity checks
    assert d["years_since_grad"].between(0, 45).all(), "years_since_grad out of range"
    assert (d["log_wage"] > 0).all(), "log_wage has non-positive values"
    assert d["degree_level"].notna().all(), "NaN in degree_level"

    return d


df = engineer_features(df)

print("Feature engineering complete.")
print(f"\nyears_since_grad stats:")
print(df["years_since_grad"].describe())
print(f"\nlog_wage stats:")
print(df["log_wage"].describe())
print(f"\nadj_wage median by degree_level:")
print(df.groupby("degree_level")["adj_wage"].median().reindex(DEGREE_ORDER))

## 4 — Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw wage distribution
axes[0].hist(df["adj_wage"].clip(upper=300_000), bins=60, edgecolor="none", alpha=0.8)
axes[0].set_title("Distribution of Salary (raw, clipped at $300k)")
axes[0].set_xlabel("Annual Salary ($)")
axes[0].set_ylabel("Count")
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x/1000:.0f}k"))

# Log-transformed wage
axes[1].hist(df["log_wage"], bins=60, edgecolor="none", alpha=0.8, color="steelblue")
axes[1].set_title("Distribution of log(Salary + 1)")
axes[1].set_xlabel("log(Salary + 1)")
axes[1].set_ylabel("Count")

plt.suptitle("ACS PUMS 2023 — Salary Distribution (employed degree holders, age 22–65)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Survey-weighted median salary by degree level
def weighted_median(values, weights):
    """Compute weighted median."""
    sorted_idx = np.argsort(values)
    vals = np.array(values)[sorted_idx]
    wts = np.array(weights)[sorted_idx]
    cumsum = np.cumsum(wts)
    cutoff = wts.sum() / 2.0
    return vals[cumsum >= cutoff][0]

deg_stats = (
    df.groupby("degree_level")
    .apply(
        lambda g: pd.Series({
            "median_salary": weighted_median(g["adj_wage"].values, g["PWGTP"].values),
            "count": len(g),
        })
    )
    .reindex(DEGREE_ORDER)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(
    deg_stats["degree_level"],
    deg_stats["median_salary"],
    color=sns.color_palette("muted", len(DEGREE_ORDER)),
)
for bar, (_, row) in zip(bars, deg_stats.iterrows()):
    ax.text(
        bar.get_width() + 2000,
        bar.get_y() + bar.get_height() / 2,
        f"${row['median_salary']:,.0f}  (n={row['count']:,.0f})",
        va="center", fontsize=10,
    )
ax.set_xlabel("Weighted Median Annual Salary ($)")
ax.set_title("Median Salary by Degree Level (survey-weighted, ACS PUMS 2023)")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x/1000:.0f}k"))
ax.set_xlim(0, deg_stats["median_salary"].max() * 1.35)
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 / Bottom 20 specific majors by median salary (bachelor's holders only)
bach = df[df["degree_level"] == "Bachelor"].copy()

major_medians = (
    bach[bach["major_name"] != "Unknown"]
    .groupby("major_name")
    .apply(lambda g: pd.Series({
        "median_salary": weighted_median(g["adj_wage"].values, g["PWGTP"].values),
        "n": len(g),
    }))
    .query("n >= 100")  # require at least 100 observations
    .sort_values("median_salary")
    .reset_index()
)

top20 = major_medians.tail(20)
bot20 = major_medians.head(20)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, data, title, color in [
    (axes[0], bot20, "Bottom 20 Majors by Median Salary", "salmon"),
    (axes[1], top20, "Top 20 Majors by Median Salary", "steelblue"),
]:
    ax.barh(data["major_name"], data["median_salary"], color=color, alpha=0.85)
    ax.set_xlabel("Median Annual Salary ($)")
    ax.set_title(title + "\n(Bachelor's holders, ACS PUMS 2023)")
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x/1000:.0f}k"))

plt.tight_layout()
plt.show()

In [ ]:
# Salary vs. years_since_grad by degree level
exp_salary = (
    df.groupby(["degree_level", "years_since_grad"])
    .apply(lambda g: weighted_median(g["adj_wage"].values, g["PWGTP"].values))
    .reset_index(name="median_salary")
)

fig, ax = plt.subplots(figsize=(12, 6))
for deg in DEGREE_ORDER:
    data = exp_salary[exp_salary["degree_level"] == deg].sort_values("years_since_grad")
    if len(data) < 5:
        continue
    # Smooth with rolling window
    smoothed = data.set_index("years_since_grad")["median_salary"].rolling(3, center=True).mean()
    ax.plot(smoothed.index, smoothed.values, label=deg, linewidth=2)

ax.set_xlabel("Years Since Graduation")
ax.set_ylabel("Weighted Median Annual Salary ($)")
ax.set_title("Salary vs. Experience by Degree Level (ACS PUMS 2023)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"${y/1000:.0f}k"))
ax.legend(title="Degree Level")
plt.tight_layout()
plt.show()

In [ ]:
# Median salary by broad major category
cat_stats = (
    df.groupby("major_category")
    .apply(lambda g: pd.Series({
        "median_salary": weighted_median(g["adj_wage"].values, g["PWGTP"].values),
        "n": len(g),
    }))
    .sort_values("median_salary")
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(cat_stats["major_category"], cat_stats["median_salary"],
        color=sns.color_palette("viridis", len(cat_stats)), alpha=0.85)
for i, (_, row) in enumerate(cat_stats.iterrows()):
    ax.text(row["median_salary"] + 500, i, f"  ${row['median_salary']:,.0f}", va="center", fontsize=9)
ax.set_xlabel("Weighted Median Annual Salary ($)")
ax.set_title("Median Salary by Major Category (all degree levels, ACS PUMS 2023)")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x/1000:.0f}k"))
ax.set_xlim(0, cat_stats["median_salary"].max() * 1.3)
plt.tight_layout()
plt.show()

## 5 — Model Training & Evaluation

We train three models on the same features and compare them:

| Model | Notes |
|-------|-------|
| `LinearRegression` | Baseline — interpretable, assumes linear relationships |
| `Ridge` | Regularized linear — handles correlated features better |
| `GradientBoostingRegressor` | Tree ensemble — captures non-linear salary curves |

**Features used:**
- `degree_level` — ordinal encoded (Associate=0 … Doctoral=4)
- `FOD1P_enc` — raw ACS major code (166 values), target-encoded
- `major_category` — broad field bucket (15 values), one-hot encoded
- `years_since_grad` — numeric with polynomial degree-2 expansion

**Target:** `log(adj_wage + 1)` — predictions are back-transformed with `expm1()` for evaluation in dollars.

In [ ]:
# ── Train / test split ────────────────────────────────────────────────────────
FEATURES = ["degree_level", "FOD1P_enc", "major_category", "years_since_grad"]
TARGET = "log_wage"

X = df[FEATURES]
y = df[TARGET]
weights = df["PWGTP"].values  # survey weights for sample_weight

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, weights, test_size=0.2, random_state=42
)

print(f"Train: {len(X_train):,} rows")
print(f"Test:  {len(X_test):,} rows")

# ── Preprocessor ──────────────────────────────────────────────────────────────
numeric_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        # Ordinal: Associate < Bachelor < Master < Professional < Doctoral
        ("degree", OrdinalEncoder(
            categories=[DEGREE_ORDER],
            handle_unknown="use_encoded_value",
            unknown_value=-1,
        ), ["degree_level"]),

        # Target encoder for specific major (166 codes) — smoothed mean encoding
        ("fod1p", TargetEncoder(target_type="continuous", smooth="auto"), ["FOD1P_enc"]),

        # One-hot for broad major category (~15 buckets)
        ("major_cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["major_category"]),

        # Polynomial expansion of years_since_grad
        ("exp", numeric_pipe, ["years_since_grad"]),
    ],
    remainder="drop",
)

# ── Model definitions ─────────────────────────────────────────────────────────
models = {
    "Linear Regression": Pipeline([
        ("pre", preprocessor),
        ("model", LinearRegression()),
    ]),
    "Ridge": Pipeline([
        ("pre", preprocessor),
        ("model", Ridge(alpha=10.0)),
    ]),
    "Gradient Boosting": Pipeline([
        ("pre", preprocessor),
        ("model", GradientBoostingRegressor(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            random_state=42,
        )),
    ]),
}

print("Preprocessor and model definitions ready.")

In [ ]:
def evaluate(name: str, pipeline, X_tr, y_tr, w_tr, X_te, y_te) -> dict:
    """Train pipeline and return evaluation metrics on the test set."""
    print(f"Training {name}...", end=" ", flush=True)
    pipeline.fit(X_tr, y_tr, model__sample_weight=w_tr)
    print("done.")

    log_pred = pipeline.predict(X_te)
    pred = np.expm1(log_pred)          # back-transform to dollars
    actual = np.expm1(y_te.values)

    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae  = mean_absolute_error(actual, pred)
    r2   = r2_score(actual, pred)
    mdape = np.median(np.abs((actual - pred) / actual)) * 100

    return {
        "Model": name,
        "RMSE ($)": rmse,
        "MAE ($)": mae,
        "R²": r2,
        "MdAPE (%)": mdape,
        "pipeline": pipeline,
    }


results = []
best_pipeline = None
for name, pipe in models.items():
    r = evaluate(name, pipe, X_train, y_train, w_train, X_test, y_test)
    results.append(r)
    if best_pipeline is None or r["R²"] > results[-2]["R²"]:
        best_pipeline = r["pipeline"]
        best_name = name

results_df = pd.DataFrame(results).drop(columns="pipeline")
pd.set_option("display.float_format", "{:.4f}".format)
print(f"\n{'='*60}")
print(f"Model Comparison (test set)")
print(f"{'='*60}")
print(results_df.to_string(index=False))
print(f"\nBest model: {best_name}")
pd.set_option("display.float_format", "{:,.0f}".format)

In [ ]:
# Actual vs. predicted scatter plot (best model)
log_pred_test = best_pipeline.predict(X_test)
pred_test = np.expm1(log_pred_test)
actual_test = np.expm1(y_test.values)

clip = 300_000
fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(
    actual_test.clip(max=clip),
    pred_test.clip(max=clip),
    alpha=0.03, s=8, color="steelblue",
)
ax.plot([0, clip], [0, clip], "r--", linewidth=1.5, label="Perfect prediction")
ax.set_xlabel("Actual Salary ($)")
ax.set_ylabel("Predicted Salary ($)")
ax.set_title(f"Actual vs. Predicted — {best_name}\n(test set, clipped at $300k)")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x/1000:.0f}k"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"${y/1000:.0f}k"))
ax.legend()
plt.tight_layout()
plt.show()

# Save best model
MODEL_PATH = "salary_model.pkl"
joblib.dump(best_pipeline, MODEL_PATH)
print(f"Best model saved to {MODEL_PATH}")

## 6 — Inference / Prediction Helper

Use `predict_salary()` to get a salary estimate for any combination of degree, major, and years of experience.

In [ ]:
# Reverse lookup: major name → FOD1P code
MAJOR_NAME_TO_CODE = {v: k for k, v in FOD1P_LABELS.items()}

def predict_salary(
    degree: str,
    major: str,
    years_since_grad: int,
    pipeline=None,
) -> float:
    """
    Predict annual salary (in dollars).

    Parameters
    ----------
    degree : str
        One of: 'Associate', 'Bachelor', 'Master', 'Professional', 'Doctoral'
    major : str
        Major name as it appears in FOD1P_LABELS, e.g. 'Computer Science'.
        Use 'Unknown' for associate holders or when major is unspecified.
    years_since_grad : int
        Approximate years since degree completion (0–45).
    pipeline : fitted sklearn Pipeline, optional
        Defaults to `best_pipeline` trained in Section 5.

    Returns
    -------
    float : predicted annual salary in dollars
    """
    if pipeline is None:
        pipeline = best_pipeline

    if degree not in DEGREE_ORDER:
        raise ValueError(f"degree must be one of {DEGREE_ORDER}")

    # Resolve FOD1P code
    fod1p_code = MAJOR_NAME_TO_CODE.get(major, 0)

    # Determine major category
    major_cat = MAJOR_CATEGORY_MAP.get(fod1p_code, "Other")

    years_clipped = max(0, min(45, years_since_grad))

    row = pd.DataFrame([{
        "degree_level": degree,
        "FOD1P_enc": fod1p_code,
        "major_category": major_cat,
        "years_since_grad": years_clipped,
    }])

    log_pred = pipeline.predict(row)[0]
    return np.expm1(log_pred)


# ── Example predictions ───────────────────────────────────────────────────────
examples = [
    ("Bachelor", "Computer Science",               0),
    ("Bachelor", "Computer Science",               5),
    ("Bachelor", "Computer Science",              10),
    ("Bachelor", "Computer Science",              20),
    ("Bachelor", "Business Management and Administration", 10),
    ("Master",   "Business Management and Administration", 10),
    ("Bachelor", "Elementary Education",           10),
    ("Doctoral",  "Electrical Engineering",         5),
    ("Bachelor", "Nursing",                        10),
    ("Master",   "Statistics and Decision Science", 5),
]

rows = []
for deg, maj, yrs in examples:
    sal = predict_salary(deg, maj, yrs)
    rows.append({"Degree": deg, "Major": maj, "Years Since Grad": yrs, "Predicted Salary": sal})

pred_df = pd.DataFrame(rows)
pred_df["Predicted Salary"] = pred_df["Predicted Salary"].map("${:,.0f}".format)
print(pred_df.to_string(index=False))

In [ ]:
# List all available major names for reference
print("Available majors (pass to predict_salary as the `major` argument):\n")
for code, name in sorted(FOD1P_LABELS.items(), key=lambda x: x[1]):
    print(f"  {name}")